# M5 Demand Forecasting - server runner

Run from the project root on the Jupyter server, top to bottom. Because the
server is firewalled from Kaggle, the data is supplied as a zip (`TIME1_m5_raw_data.zip`)
rather than downloaded here.

Copy the printed output of the EDA, baseline, sensitivity, LightGBM, and Chronos
cells back into the chat.

## 1. Confirm we are in the project root

In [ ]:
import os, sys
# If needed: os.chdir('/home/25m1501/TIme/TIME1')
print('Working directory:', os.getcwd())
assert os.path.isdir('src'), 'Not in project root: no src/ folder. Use os.chdir() above.'

## 2. Install dependencies

In [ ]:
!{sys.executable} -m pip install -q -r requirements.txt

## 3. Unzip the supplied M5 data

Upload `TIME1_m5_raw_data.zip` into this folder first. The Kaggle download step
is intentionally skipped: the server cannot reach Kaggle.

In [ ]:
import glob, zipfile
zip_path = 'TIME1_m5_raw_data.zip'
if not os.path.exists('data/raw/calendar.csv'):
    assert os.path.exists(zip_path), f'{zip_path} not found in {os.getcwd()}; upload it first.'
    with zipfile.ZipFile(zip_path) as z:
        z.extractall('.')
print('Files in data/raw/:')
for p in sorted(glob.glob('data/raw/*.csv')):
    print(f'  {p}  ({os.path.getsize(p)/1e6:.1f} MB)')

## 4. Prepare and validate the data

In [ ]:
!{sys.executable} -m src.data.load
!{sys.executable} -m src.data.validate

## 5. EDA - paste this output back

In [ ]:
!{sys.executable} -m src.eda

In [ ]:
from IPython.display import Image, display
for path in sorted(glob.glob('reports/figures/*.png')):
    print(path); display(Image(path))

## 6. Baseline backtest - paste this table back

In [ ]:
!{sys.executable} -m scripts.run_baselines

## 7. Backtest-window sensitivity check - paste this table back

In [ ]:
!{sys.executable} -m scripts.window_sensitivity --windows 3 5 6

## 8. LightGBM: tune with Optuna, then backtest

Compute-heavy. Run the sanity line first, then the tuned line. Paste both back.

In [ ]:
!{sys.executable} -m scripts.run_lightgbm --no-tune

In [ ]:
!{sys.executable} -m scripts.run_lightgbm --trials 50

## 9. Chronos foundation model (Phase 3)

Install the optional dependencies (torch + chronos-forecasting), then backtest
Chronos zero-shot. This benefits from a GPU; on the first run it downloads the
pretrained weights. It also reports the probabilistic pinball loss. Paste the
printed summary back.

In [ ]:
!{sys.executable} -m pip install -q -r requirements-optional.txt
import torch
print('CUDA available:', torch.cuda.is_available(),
      '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

In [ ]:
# chronos-t5-small by default (see config). chronos-bolt models are faster:
# add --model amazon/chronos-bolt-small for a quicker run.
!{sys.executable} -m scripts.run_chronos

## 10. (Optional) unit tests

In [ ]:
!{sys.executable} -m pytest -q